# What is a quarantine dataset used for?

Quarantine means separating data that has a problem from the valid data, so the bad data does not affect the main processing

Incoming Data
     ↓
   Validate
     ↓
 ┌───────────────┐
 │               │
Valid Data     Invalid Data
   ↓                ↓
Silver/Gold      Quarantine

A quarantine dataset is a separate location where invalid or suspicious records are stored instead of allowing them into the main pipeline . it helps us preserve bad records for investigation , correction and possible reprocessing without affecting the valid data



#How does a parse error differ from an invalid business value?

Parse error : a parse error happens when the spark cannot read /understand the structure ir format of the incoming data

Invalid Bussiness Value : the data can be read successfully but the values cannot align with
the business rules

A parse error occurs when the data cannot be correctly interpreted according to the expected format or schema. An invalid business value occurs when the data is successfully parsed but violates a business rule.



#Why normalize email and city values?

Normalize means converting data into a consistent format before storing or comparing it
it is important mainly to detect duplication , accurate joins , consistent reporting , data quality and searching / filtering

We normalize email and city values to make them consistent, which helps prevent duplicates, improve joins and matching, and produce accurate reporting and data quality.

#What do trim and lower do?

Trim() : is used to remove left and right spaces in a value and make them valid by using memory consistently

Lower(): converts the specific value or record into lower case to validate the data easily

#What does coalesce return when several columns are null?

coalesce() : is used to handle null values so if there are several null values it keeps checking the columns from left  tto right until it finds a non null value

#Why is a missing email allowed but a missing line_id rejected?

“A missing email is allowed because email is an optional business attribute, whereas line_id is a mandatory identifier used to uniquely identify and track each record. Therefore, a missing line_id is treated as a data-quality failure and the record is rejected.


#1. What is a referential-integrity violation?

A referential integrity violation happens when a record in a child table refers to a record that does not exists in the parent table .

A referential-integrity violation occurs when a foreign key in a child table references a primary key that does not exist in the parent table. For example, if an order contains customer_id 999 but customer 999 does not exist in the customer master, that order violates referential integrity.



#Why retain more than one rejection reason per row?

We retain multiple rejection reasons because a single record can violate multiple validation rules . Capturing all applicable reasons improves debugging , data correction ,data quality reporting and prevent repeated processing of same bad recoord


#What is the accepted-row grain before deduplication?

Grain means what one row represents in a dataset.
So, accepted-row grain before deduplication

Accepted row grain defines what one valid row represents before duplication . For example, if the grain is one row per customer , then each accepted row represents a customer , and duplicate customer reccords can later be  identified using the customer id .


#Why preserve original values alongside cleaned values?

We preserve original values alongside cleaned values to maintain source fidelity, support auditing and debugging, provide data lineage, and allow reprocessing if the cleaning rules change.


#Hands-on questions (11-20)


In [1]:
pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [3]:
spark=SparkSession.builder.appName("cleaning and quarentine").getOrCreate()

In [65]:
customers=[(101,"Ramya"," Ramya@gmail.com ","HYDERBAD","2024-01-15",True),
            (102,"Anil",None,"Bangalore","2024-02-20",True),
            (103,"Sita"," sita@gmail.com ","  Chennai  ","2024-03-10",False),
            (104,"Ravi","ravi@gmail.com","Mumbai","2024-04-05",True),
            (101,"Priya","priya@gmail.com"," Delhi","2024-05-12",True),
            (105,"Kiran","KIRAN39@GMAIL.COM","Pune","2024-06-18",False),
            (104,"Meena"," meena@gmail.com  ","KOLKATA","2024-07-22",True),
            (108,"Arjun"," ","Hyderabad","2024-08-09",True),
            (109,"Divya","divya@gmail.com","BangalORE","2024-09-14",True),
            (103,"Varun","VaRUn400@Outook.Com",None,"2024-10-25",False)
]
schema=StructType([
    StructField("customer_id",IntegerType(),True),
    StructField("first_name",StringType(),True),
    StructField("email",StringType(),True),
    StructField("city",StringType(),True),
    StructField("order_date",StringType(),True),
    StructField("is_active",BooleanType())
])

df_customers=spark.createDataFrame(customers,schema)
df_customers.show(truncate=False)

+-----------+----------+-------------------+-----------+----------+---------+
|customer_id|first_name|email              |city       |order_date|is_active|
+-----------+----------+-------------------+-----------+----------+---------+
|101        |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |
|102        |Anil      |NULL               |Bangalore  |2024-02-20|true     |
|103        |Sita      | sita@gmail.com    |  Chennai  |2024-03-10|false    |
|104        |Ravi      |ravi@gmail.com     |Mumbai     |2024-04-05|true     |
|101        |Priya     |priya@gmail.com    | Delhi     |2024-05-12|true     |
|105        |Kiran     |KIRAN39@GMAIL.COM  |Pune       |2024-06-18|false    |
|104        |Meena     | meena@gmail.com   |KOLKATA    |2024-07-22|true     |
|108        |Arjun     |                   |Hyderabad  |2024-08-09|true     |
|109        |Divya     |divya@gmail.com    |BangalORE  |2024-09-14|true     |
|103        |Varun     |VaRUn400@Outook.Com|NULL       |2024-10-

In [5]:
# #Normalize customer email and city without changing IDs.
df_customers=df_customers.withColumn("email", lower(trim(F.col("email"))))\
                         .withColumn("city",initcap(trim("city")))
df_customers.show(truncate=False)

+-----------+----------+-------------------+---------+----------+---------+
|customer_id|first_name|email              |city     |order_date|is_active|
+-----------+----------+-------------------+---------+----------+---------+
|101        |Ramya     |ramya@gmail.com    |Hyderbad |2024-01-15|true     |
|102        |Anil      |NULL               |Bangalore|2024-02-20|true     |
|103        |Sita      |sita@gmail.com     |Chennai  |2024-03-10|false    |
|104        |Ravi      |ravi@gmail.com     |Mumbai   |2024-04-05|true     |
|101        |Priya     |priya@gmail.com    |Delhi    |2024-05-12|true     |
|105        |Kiran     |kiran39@gmail.com  |Pune     |2024-06-18|false    |
|104        |Meena     |meena@gmail.com    |Kolkata  |2024-07-22|true     |
|108        |Arjun     |                   |Hyderabad|2024-08-09|true     |
|109        |Divya     |divya@gmail.com    |Bangalore|2024-09-14|true     |
|103        |Varun     |varun400@outook.com|NULL     |2024-10-25|false    |
+-----------

In [6]:
# Flag null and blank line_id values using a tiny fixture.
customers = [
    (101, "L001"),
    (102, None),
    (103, ""),
    (104, "L004"),
    (105, "   ")
]
df_small_customers=spark.createDataFrame(customers,["customer_id","line_id"])
df_small_customers.show()
df_small_customers=df_small_customers.withColumn("line_id_invalid",
                                                when (
                                                    col("line_id").isNull() | (trim(col("line_id"))==""),True)
                                                .otherwise(False)
                                                )
df_small_customers.show()

+-----------+-------+
|customer_id|line_id|
+-----------+-------+
|        101|   L001|
|        102|   NULL|
|        103|       |
|        104|   L004|
|        105|       |
+-----------+-------+

+-----------+-------+---------------+
|customer_id|line_id|line_id_invalid|
+-----------+-------+---------------+
|        101|   L001|          false|
|        102|   NULL|           true|
|        103|       |           true|
|        104|   L004|          false|
|        105|       |           true|
+-----------+-------+---------------+



In [7]:
from decimal import Decimal

In [8]:
order_items = [
    (1001, 5001, 201, "Laptop Stand",        -2, Decimal("-499.99"),  Decimal("10.50")),
    (1002, 5001, 202, "Wireless Keyboard",  None, Decimal("1299.75"), Decimal("50.25")),
    (1003, 5002, 203, "USB Cable",          -5, Decimal("99.99"),   Decimal("5.00")),
    (1004, 5002, 204, "Wireless Mouse",     3, Decimal("250.50"),  Decimal("-15.75")),
    (1005, 5003, 205, "Webcam",             1, Decimal("999.99"),  Decimal("0.00")),
    (1006, 5003, 201, "Laptop Stand",       4, Decimal("499.99"),  Decimal("25.50")),
    (1007, 5004, 206, "Monitor",            2, Decimal("750.25"),  Decimal("20.00")),
    (1008, 5005, 207, "Mechanical Keyboard",1, Decimal("1599.90"), Decimal("100.50")),
    (1009, 5005, 208, "Mouse Pad",          6, Decimal("-75.25"),   Decimal("5.25")),
    (1010, 5006, 209, "USB Hub",            2, Decimal("349.99"),  None),
    (1011, 5007, 210, "Headphones",         3, Decimal("450.75"),  Decimal("30.25")),
    (1012, 5008, 203, "USB Cable",          2, Decimal("99.99"),   Decimal("-10.00")),
    (1013, 5009, 211, "4K Monitor",         -1, Decimal("2499.99"), Decimal("150.75")),
    (1014, 5010, 212, "Webcam Cover",       5, Decimal("-125.50"),  Decimal("12.50")),
    (1015, 5010, 201, "Laptop Stand",       None, Decimal("499.99"),  Decimal("50.00"))
]

schema=StructType([
    StructField("order_item_id",IntegerType(),False),
    StructField("order_id",IntegerType(),True),
    StructField("product_id",IntegerType(),True),
    StructField("product_name",StringType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DecimalType(10,2),True),
    StructField("discount",DecimalType(10,2),True)
])
df_order_items=spark.createDataFrame(order_items,schema)
df_order_items.show()


+-------------+--------+----------+-------------------+--------+----------+--------+
|order_item_id|order_id|product_id|       product_name|quantity|unit_price|discount|
+-------------+--------+----------+-------------------+--------+----------+--------+
|         1001|    5001|       201|       Laptop Stand|      -2|   -499.99|   10.50|
|         1002|    5001|       202|  Wireless Keyboard|    NULL|   1299.75|   50.25|
|         1003|    5002|       203|          USB Cable|      -5|     99.99|    5.00|
|         1004|    5002|       204|     Wireless Mouse|       3|    250.50|  -15.75|
|         1005|    5003|       205|             Webcam|       1|    999.99|    0.00|
|         1006|    5003|       201|       Laptop Stand|       4|    499.99|   25.50|
|         1007|    5004|       206|            Monitor|       2|    750.25|   20.00|
|         1008|    5005|       207|Mechanical Keyboard|       1|   1599.90|  100.50|
|         1009|    5005|       208|          Mouse Pad|       6| 

In [9]:
# Reject nonpositive quantities and negative prices.
df_order_postive=df_order_items.filter((col("quantity")>0) & (col("unit_price")>0))
df_order_postive.show()

+-------------+--------+----------+-------------------+--------+----------+--------+
|order_item_id|order_id|product_id|       product_name|quantity|unit_price|discount|
+-------------+--------+----------+-------------------+--------+----------+--------+
|         1004|    5002|       204|     Wireless Mouse|       3|    250.50|  -15.75|
|         1005|    5003|       205|             Webcam|       1|    999.99|    0.00|
|         1006|    5003|       201|       Laptop Stand|       4|    499.99|   25.50|
|         1007|    5004|       206|            Monitor|       2|    750.25|   20.00|
|         1008|    5005|       207|Mechanical Keyboard|       1|   1599.90|  100.50|
|         1010|    5006|       209|            USB Hub|       2|    349.99|    NULL|
|         1011|    5007|       210|         Headphones|       3|    450.75|   30.25|
|         1012|    5008|       203|          USB Cable|       2|     99.99|  -10.00|
+-------------+--------+----------+-------------------+--------+-

In [10]:
# Reject missing discounts and values outside the inclusive valid range.
df_reject_missing=df_order_items.filter((col("discount")>=0) & (col("discount")<=200) & (col("discount").isNotNull()))
df_reject_missing.show()

+-------------+--------+----------+-------------------+--------+----------+--------+
|order_item_id|order_id|product_id|       product_name|quantity|unit_price|discount|
+-------------+--------+----------+-------------------+--------+----------+--------+
|         1001|    5001|       201|       Laptop Stand|      -2|   -499.99|   10.50|
|         1002|    5001|       202|  Wireless Keyboard|    NULL|   1299.75|   50.25|
|         1003|    5002|       203|          USB Cable|      -5|     99.99|    5.00|
|         1005|    5003|       205|             Webcam|       1|    999.99|    0.00|
|         1006|    5003|       201|       Laptop Stand|       4|    499.99|   25.50|
|         1007|    5004|       206|            Monitor|       2|    750.25|   20.00|
|         1008|    5005|       207|Mechanical Keyboard|       1|   1599.90|  100.50|
|         1009|    5005|       208|          Mouse Pad|       6|    -75.25|    5.25|
|         1011|    5007|       210|         Headphones|       3| 

In [11]:
order = [
    (50001, 10001, 1001, 2, 45000.00, 5.0, 85500.00),
    (50002, 10001, 1005, 1, 12000.00, 10.0, 10800.00),
    (50003, 10002, 1003, 3, 2500.00, 5.0, 7125.00),
    (50004, 10002, 1008, 1, 18500.00, 15.0, 15725.00),
    (50005, 10003, 1002, 2, 32000.00, 8.0, 58880.00),
    (50006, 10003, 1006, 4, 1500.00, 5.0, 5700.00),
    (50007, 10004, 1004, 1, 55000.00, 10.0, 49500.00),
    (50008, 10004, 1007, 2, 7500.00, 5.0, 14250.00),
    (50009, 10005, 1009, 3, 2200.00, 10.0, 5940.00),
    (50010, 10006, 1010, 1, 48000.00, 12.0, 42240.00),
    (50011, 10007, 1003, 2, 2500.00, 5.0, 4750.00),
    (50012, 10008, 1005, 2, 12000.00, 7.5, 22200.00),
    (50013, 10009, 1001, 1, 45000.00, 10.0, 40500.00),
    (50014, 10010, 1006, 5, 1500.00, 5.0, 7125.00),
    (50015, 10011, 1008, 2, 18500.00, 10.0, 33300.00),
    (50016, 10012, 1002, 1, 32000.00, 15.0, 27200.00),
    (50017, 10013, 1004, 2, 55000.00, 5.0, 104500.00),
    (50018, 10014, 1007, 3, 7500.00, 8.0, 20700.00),
    (50019, 10015, 1009, 2, 2200.00, 5.0, 4180.00),
    (50020, 10015, 1010, 1, 48000.00, 10.0, 43200.00),
    (50021, 10021, 1021, 1, 48000.00, 10.0, 43200.00),
    (50022, 10022, 1022, 1, 48000.00, 10.0, 43200.00)
]

schema=StructType([
    StructField("order_item_id",IntegerType(),True),
    StructField("order_id",IntegerType(),True),
    StructField("product_id",IntegerType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DoubleType(),True),
    StructField("discount_pct",DoubleType(),True),
    StructField("item_amount",DoubleType(),True)
])

df_orders=spark.createDataFrame(order,schema)
df_orders.show()

+-------------+--------+----------+--------+----------+------------+-----------+
|order_item_id|order_id|product_id|quantity|unit_price|discount_pct|item_amount|
+-------------+--------+----------+--------+----------+------------+-----------+
|        50001|   10001|      1001|       2|   45000.0|         5.0|    85500.0|
|        50002|   10001|      1005|       1|   12000.0|        10.0|    10800.0|
|        50003|   10002|      1003|       3|    2500.0|         5.0|     7125.0|
|        50004|   10002|      1008|       1|   18500.0|        15.0|    15725.0|
|        50005|   10003|      1002|       2|   32000.0|         8.0|    58880.0|
|        50006|   10003|      1006|       4|    1500.0|         5.0|     5700.0|
|        50007|   10004|      1004|       1|   55000.0|        10.0|    49500.0|
|        50008|   10004|      1007|       2|    7500.0|         5.0|    14250.0|
|        50009|   10005|      1009|       3|    2200.0|        10.0|     5940.0|
|        50010|   10006|    

In [12]:
products = [
    (1001, "Laptop", "Electronics", "Computers", "Dell", 201, 42000.00, 45000.00, "2024-01-15", 4.5, 25),
    (1002, "Smartphone", "Electronics", "Mobiles", "Samsung", 202, 28000.00, 32000.00, "2024-02-10", 4.4, 40),
    (1003, "Headphones", "Electronics", "Audio", "Sony", 203, 1800.00, 2500.00, "2024-03-05", 4.3, 75),
    (1004, "Television", "Electronics", "TV", "LG", 204, 48000.00, 55000.00, "2024-04-20", 4.6, 18),
    (1005, "Smart Watch", "Electronics", "Wearables", "Apple", 205, 10000.00, 12000.00, "2024-05-12", 4.7, 30),
    (1006, "Wireless Mouse", "Electronics", "Accessories", "Logitech", 206, 1100.00, 1500.00, "2024-06-18", 4.2, 90),
    (1007, "Keyboard", "Electronics", "Accessories", "HP", 207, 5500.00, 7500.00, "2024-07-25", 4.1, 55),
    (1008, "Tablet", "Electronics", "Tablets", "Lenovo", 208, 16000.00, 18500.00, "2024-08-14", 4.4, 35),
    (1009, "Bluetooth Speaker", "Electronics", "Audio", "JBL", 209, 1700.00, 2200.00, "2024-09-08", 4.5, 65),
    (1010, "Gaming Console", "Electronics", "Gaming", "Sony", 203, 42000.00, 48000.00, "2024-10-22", 4.8, 12),
    (1011, "Office Chair", "Furniture", "Chairs", "GreenSoul", 210, 7000.00, 9500.00, "2024-11-10", 4.3, 20),
    (1012, "Study Table", "Furniture", "Tables", "Wakefit", 211, 6500.00, 8500.00, "2024-12-05", 4.2, 15),
    (1013, "Backpack", "Fashion", "Bags", "American Tourister", 212, 1800.00, 2500.00, "2025-01-18", 4.4, 50),
    (1014, "Running Shoes", "Fashion", "Footwear", "Nike", 213, 4500.00, 6000.00, "2025-02-14", 4.6, 45),
    (1015, "Coffee Maker", "Home Appliances", "Kitchen", "Philips", 214, 3500.00, 4500.00, "2025-03-20", 4.3, 28),
    (1016, "Air Fryer", "Home Appliances", "Kitchen", "Prestige", 215, 5000.00, 6500.00, "2025-04-12", 4.5, 22),
    (1017, "Water Bottle", "Home & Kitchen", "Drinkware", "Milton", 216, 500.00, 800.00, "2025-05-08", 4.2, 100),
    (1018, "Desk Lamp", "Home & Kitchen", "Lighting", "Philips", 214, 900.00, 1300.00, "2025-06-15", 4.1, 60),
    (1019, "Power Bank", "Electronics", "Accessories", "Mi", 217, 900.00, 1400.00, "2025-07-10", 4.4, 80),
    (1020, "USB Cable", "Electronics", "Accessories", "Anker", 218, 500.00, 900.00, "2025-08-25", 4.3, 120)
]

schema= StructType([
    StructField("product_id",IntegerType(),True),
    StructField("product_name",StringType(),True),
    StructField("category",StringType(),True),
    StructField("sub_category",StringType(),True),
    StructField("brand",StringType(),True),
    StructField("supplier_id",IntegerType(),True),
    StructField("unit_cost",DoubleType(),True),
    StructField("selling_price",DoubleType(),True),
    StructField("launch_date",StringType(),True),
    StructField("product_rating",DoubleType(),True),
    StructField("stock_quantity",IntegerType(),True)
])
df_products=spark.createDataFrame(products,schema)

In [13]:
# Find unknown product IDs with a left anti join.
df_unknow=df_orders.join(df_products,df_orders.product_id == df_products.product_id,"left_anti")
df_unknow.show()

+-------------+--------+----------+--------+----------+------------+-----------+
|order_item_id|order_id|product_id|quantity|unit_price|discount_pct|item_amount|
+-------------+--------+----------+--------+----------+------------+-----------+
|        50021|   10021|      1021|       1|   48000.0|        10.0|    43200.0|
|        50022|   10022|      1022|       1|   48000.0|        10.0|    43200.0|
+-------------+--------+----------+--------+----------+------------+-----------+



In [14]:
# Find unknown order IDs independently of product validation.
order = [
    (50001, 10001, 1001, 2, 45000.00, 5.0, 85500.00),
    (50002, 10001, 1005, 1, 12000.00, 10.0, 10800.00),
    (50003, 10002, 1003, 3, 2500.00, 5.0, 7125.00),
    (50004, 10002, 1008, 1, 18500.00, 15.0, 15725.00),
    (50005, 10003, 1002, 2, 32000.00, 8.0, 58880.00),
    (50006, 10003, 1006, 4, 1500.00, 5.0, 5700.00),
    (50007, 10004, 1004, 1, 55000.00, 10.0, 49500.00),
    (50008, 10004, 1007, 2, 7500.00, 5.0, 14250.00),
    (50009, 10005, 1009, 3, 2200.00, 10.0, 5940.00),
    (50010, 10006, 1010, 1, 48000.00, 12.0, 42240.00),
    (50011, 10007, 1003, 2, 2500.00, 5.0, 4750.00),
    (50012, 10008, 1005, 2, 12000.00, 7.5, 22200.00),
    (50013, 10009, 1001, 1, 45000.00, 10.0, 40500.00),
    (50014, 10010, 1006, 5, 1500.00, 5.0, 7125.00),
    (50015, 10011, 1008, 2, 18500.00, 10.0, 33300.00),
    (50016, 10012, 1002, 1, 32000.00, 15.0, 27200.00),
    (50017, 10013, 1004, 2, 55000.00, 5.0, 104500.00),
    (50018, 10014, 1007, 3, 7500.00, 8.0, 20700.00),
    (50019, 10015, 1009, 2, 2200.00, 5.0, 4180.00),
    (50020, 10015, 1010, 1, 48000.00, 10.0, 43200.00),
    (50021, 10021, 1021, 1, 48000.00, 10.0, 43200.00),
    (50022, 10022, 1022, 1, 48000.00, 10.0, 43200.00)
]

schema=StructType([
    StructField("order_item_id",IntegerType(),True),
    StructField("order_id",IntegerType(),True),
    StructField("product_id",IntegerType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DoubleType(),True),
    StructField("discount_pct",DoubleType(),True),
    StructField("item_amount",DoubleType(),True)
])

df_orders=spark.createDataFrame(order,schema)
# df_orders.show()

reference_orders = [
    (10001,),
    (10002,),
    (10003,),
    (10004,),
    (10005,),
    (10006,),
    (10007,),
    (10008,),
    (10009,),
    (10010,),
    (10011,),
    (10012,),
    (10013,),
    (10014,),
    (10015,)
]
reference_schema = StructType([
    StructField("order_id",IntegerType(),True)
])

df_refernce_order=spark.createDataFrame(reference_orders,reference_schema)
df_unknown_orders=df_orders.join(df_refernce_order,df_orders.order_id == df_refernce_order.order_id,"left_anti")
df_unknown_orders.show()

#independently of product validation.

reference_products = [
    (1001,),
    (1002,),
    (1003,),
    (1004,),
    (1005,),
    (1006,),
    (1007,),
    (1008,),
    (1009,),
    (1010,)
]

reference_product_schema = StructType([
    StructField("product_id", IntegerType(), True)
])

df_reference_products = spark.createDataFrame(reference_products,reference_product_schema)
df_independent_product=df_products.join(df_reference_products,df_products.product_id == df_reference_products.product_id,"left_anti")
df_independent_product.show()

+-------------+--------+----------+--------+----------+------------+-----------+
|order_item_id|order_id|product_id|quantity|unit_price|discount_pct|item_amount|
+-------------+--------+----------+--------+----------+------------+-----------+
|        50022|   10022|      1022|       1|   48000.0|        10.0|    43200.0|
|        50021|   10021|      1021|       1|   48000.0|        10.0|    43200.0|
+-------------+--------+----------+--------+----------+------------+-----------+

+----------+-------------+---------------+------------+------------------+-----------+---------+-------------+-----------+--------------+--------------+
|product_id| product_name|       category|sub_category|             brand|supplier_id|unit_cost|selling_price|launch_date|product_rating|stock_quantity|
+----------+-------------+---------------+------------+------------------+-----------+---------+-------------+-----------+--------------+--------------+
|      1016|    Air Fryer|Home Appliances|     Kitchen

In [15]:
# Build an array containing every applicable rejection reason.
df_customers.show(truncate=False)
df_array=df_customers.withColumn("rejection_reason",
                                  expr("""filter(
                                 array(
                                      CASE WHEN email IS NULL or trim(email) = '' THEN 'missing_email' END,
                                      CASE when city is Null then 'missing_city' END

),  x -> x IS NOT NULL
)
"""))
df_array.show(truncate=False)

+-----------+----------+-------------------+---------+----------+---------+
|customer_id|first_name|email              |city     |order_date|is_active|
+-----------+----------+-------------------+---------+----------+---------+
|101        |Ramya     |ramya@gmail.com    |Hyderbad |2024-01-15|true     |
|102        |Anil      |NULL               |Bangalore|2024-02-20|true     |
|103        |Sita      |sita@gmail.com     |Chennai  |2024-03-10|false    |
|104        |Ravi      |ravi@gmail.com     |Mumbai   |2024-04-05|true     |
|101        |Priya     |priya@gmail.com    |Delhi    |2024-05-12|true     |
|105        |Kiran     |kiran39@gmail.com  |Pune     |2024-06-18|false    |
|104        |Meena     |meena@gmail.com    |Kolkata  |2024-07-22|true     |
|108        |Arjun     |                   |Hyderabad|2024-08-09|true     |
|109        |Divya     |divya@gmail.com    |Bangalore|2024-09-14|true     |
|103        |Varun     |varun400@outook.com|NULL     |2024-10-25|false    |
+-----------

In [16]:
df_customers.show()

+-----------+----------+-------------------+---------+----------+---------+
|customer_id|first_name|              email|     city|order_date|is_active|
+-----------+----------+-------------------+---------+----------+---------+
|        101|     Ramya|    ramya@gmail.com| Hyderbad|2024-01-15|     true|
|        102|      Anil|               NULL|Bangalore|2024-02-20|     true|
|        103|      Sita|     sita@gmail.com|  Chennai|2024-03-10|    false|
|        104|      Ravi|     ravi@gmail.com|   Mumbai|2024-04-05|     true|
|        101|     Priya|    priya@gmail.com|    Delhi|2024-05-12|     true|
|        105|     Kiran|  kiran39@gmail.com|     Pune|2024-06-18|    false|
|        104|     Meena|    meena@gmail.com|  Kolkata|2024-07-22|     true|
|        108|     Arjun|                   |Hyderabad|2024-08-09|     true|
|        109|     Divya|    divya@gmail.com|Bangalore|2024-09-14|     true|
|        103|     Varun|varun400@outook.com|     NULL|2024-10-25|    false|
+-----------

In [17]:
# import shutil
# shutil.rmtree("data.quarantine")

In [18]:
# 11. Write accepted and quarantined rows to separate Parquet paths.
df_accepted=df_customers.filter((col("email").isNotNull()) & (trim("email") != '') & (col("city").isNotNull()))
df_accepted.show(truncate=False)
df_accepted.write.parquet("data/accepted",mode="overwrite")
df_accepted=spark.read.parquet("data/accepted")
df_quarantine = df_customers.filter((col("email").isNull()) | (trim("email") == '')  | (col("city").isNull()))
df_quarantine.show(truncate = False)
df_quarantine.write.parquet("data/quarantine",mode="overwrite")
df_quarantine=spark.read.parquet("data/quarantine")

+-----------+----------+-----------------+---------+----------+---------+
|customer_id|first_name|email            |city     |order_date|is_active|
+-----------+----------+-----------------+---------+----------+---------+
|101        |Ramya     |ramya@gmail.com  |Hyderbad |2024-01-15|true     |
|103        |Sita      |sita@gmail.com   |Chennai  |2024-03-10|false    |
|104        |Ravi      |ravi@gmail.com   |Mumbai   |2024-04-05|true     |
|101        |Priya     |priya@gmail.com  |Delhi    |2024-05-12|true     |
|105        |Kiran     |kiran39@gmail.com|Pune     |2024-06-18|false    |
|104        |Meena     |meena@gmail.com  |Kolkata  |2024-07-22|true     |
|109        |Divya     |divya@gmail.com  |Bangalore|2024-09-14|true     |
+-----------+----------+-----------------+---------+----------+---------+

+-----------+----------+-------------------+---------+----------+---------+
|customer_id|first_name|email              |city     |order_date|is_active|
+-----------+----------+---------

In [19]:
# Verify that raw rows equal accepted-before-dedup plus quarantined rows.
raw_count=df_customers.count()
accepted_count=df_accepted.count()
quarantine_count=df_quarantine.count()
print("Raw Rows:",raw_count)
print("Accepted before dedup:",accepted_count)
print("Quarantine:",quarantine_count)
print("Accepted+Quarantined:",accepted_count+quarantine_count)
if raw_count == accepted_count+quarantine_count:
  print("Validation passed : all rows are accounted for")
else:
  print("Validation failed : row count mismatch")

Raw Rows: 10
Accepted before dedup: 7
Quarantine: 3
Accepted+Quarantined: 10
Validation passed : all rows are accounted for


#Advanced challenges (21-30)


In [20]:
# Create a row violating quantity, price, and product rules simultaneously.
invalid_row = [(50023, 10023, 1021, -1, -100.0, 5.0, -100.0)]
df_invalid = spark.createDataFrame(invalid_row,schema)
df_invalid.show()

+-------------+--------+----------+--------+----------+------------+-----------+
|order_item_id|order_id|product_id|quantity|unit_price|discount_pct|item_amount|
+-------------+--------+----------+--------+----------+------------+-----------+
|        50023|   10023|      1021|      -1|    -100.0|         5.0|     -100.0|
+-------------+--------+----------+--------+----------+------------+-----------+



In [21]:
# Explain why summing per-rule rejection counts can exceed the quarantine row count.
from pyspark.sql.functions import col

# Rule 1: quantity violation
quantity_rejected = df_orders.filter(col("quantity") <= 0).count()

# Rule 2: price violation
price_rejected = df_orders.filter(col("unit_price") < 0).count()

# Rule 3: product violation
product_rejected = df_orders.join(df_reference_products,df_orders.product_id == df_reference_products.product_id,"left_anti").count()

# Total quarantine rows
quarantine_count = df_quarantine.count()

print("Quantity violations:", quantity_rejected)
print("Price violations:", price_rejected)
print("Product violations:", product_rejected)

print("Sum of rule violations:",
      quantity_rejected + price_rejected + product_rejected)

print("Quarantine row count:", quarantine_count)

Quantity violations: 0
Price violations: 0
Product violations: 2
Sum of rule violations: 2
Quarantine row count: 3


In [22]:
#Design mutually exclusive severity categories without losing detailed reasons.

data = [
    (101, 5, 100.0, "valid",    "[]"),
    (102, 0, 100.0, "valid",    '["invalid_quantity"]'),
    (103, 5, -50.0, "valid",    '["negative_price"]'),
    (104, 5, 100.0, "unknown",  '["unknown_product"]'),
    (105, 0, -50.0, "unknown",  '["invalid_quantity","negative_price","unknown_product"]')
]

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("product_status", StringType(), True),
    StructField("rejection_reasons", StringType(), True)
])

df_data = spark.createDataFrame(data, schema)
df_data = df_data.withColumn(
    "severity",
    when(col("rejection_reasons") =="[]","VALID")
    .when(col("rejection_reasons") == '["invalid_quantity"]' , "LOW")
    .when(col("rejection_reasons") =='["negative_price"]',"MEDIUM")
    .when(col("rejection_reasons") == '["unknown_product"]',"MEDIUM")
    .otherwise("HIGH"))

df_data.show()


+--------+--------+-----+--------------+--------------------+--------+
|order_id|quantity|price|product_status|   rejection_reasons|severity|
+--------+--------+-----+--------------+--------------------+--------+
|     101|       5|100.0|         valid|                  []|   VALID|
|     102|       0|100.0|         valid|["invalid_quantity"]|     LOW|
|     103|       5|-50.0|         valid|  ["negative_price"]|  MEDIUM|
|     104|       5|100.0|       unknown| ["unknown_product"]|  MEDIUM|
|     105|       0|-50.0|       unknown|["invalid_quantit...|    HIGH|
+--------+--------+-----+--------------+--------------------+--------+



In [23]:
# from typing import cast
#Detect text that failed numeric casting rather than confusing it with original null.
data = [
    (101, "10"),
    (102, None),
    (103, "abc"),
    (104, "25"),
    (105, "xyz"),
    (106, "0"),
    (107, ""),
    (108, "15.5"),
    (109, None),
    (110, "-5")
]
schema=StructType([
    StructField("customer_id",IntegerType(),True),
    StructField("quantity",StringType(),True)
])
df_data1=spark.createDataFrame(data,schema)
df_data2=df_data1.withColumn("quantity_raw",col("quantity"))\
                .withColumn("quantity_int",col("quantity").try_cast("int"))\
                .withColumn("status",
                            when(col("quantity_raw").isNull(),"original_missing")
                            .when(col("quantity_raw")=="","empty_string")
                            .when(col("quantity_int").isNull(),"cast_failed")
                            .otherwise("Valid")
                )
df_data2.show(truncate=False)


+-----------+--------+------------+------------+----------------+
|customer_id|quantity|quantity_raw|quantity_int|status          |
+-----------+--------+------------+------------+----------------+
|101        |10      |10          |10          |Valid           |
|102        |NULL    |NULL        |NULL        |original_missing|
|103        |abc     |abc         |NULL        |cast_failed     |
|104        |25      |25          |25          |Valid           |
|105        |xyz     |xyz         |NULL        |cast_failed     |
|106        |0       |0           |0           |Valid           |
|107        |        |            |NULL        |empty_string    |
|108        |15.5    |15.5        |NULL        |cast_failed     |
|109        |NULL    |NULL        |NULL        |original_missing|
|110        |-5      |-5          |-5          |Valid           |
+-----------+--------+------------+------------+----------------+



In [24]:
# Explain how SQL three-valued logic can accidentally admit invalid rows.
data = [
    (101, 10),
    (102, 0),
    (103, -5),
    (104, None),
    (105, 25)
]
df_data3=spark.createDataFrame(data,["customer_id","quantity"])
df_data3.printSchema()
# df_data3_filter=df_data3.filter((col("quantity").isNotNull())  & (col("quantity")>0))
# df_data3_filter.show()
df_data3.filter(~(col("quantity") <= 0)).show()

root
 |-- customer_id: long (nullable = true)
 |-- quantity: long (nullable = true)

+-----------+--------+
|customer_id|quantity|
+-----------+--------+
|        101|      10|
|        105|      25|
+-----------+--------+



In [40]:
# Test all-valid, all-invalid, and empty input DataFrames.

data_valid = [
    (101, 10),
    (102, 25),
    (103,5)
]
df_data_valid=spark.createDataFrame(data_valid,["customer_id","quantity"])
data_invalid=[
    (104,0),
    (105,-5),
    (106,None)
]
df_data_invalid=spark.createDataFrame(data_invalid,["customer_id","quantity"])

data_empty=[]
schema=StructType([
    StructField("customer_id",IntegerType(),True),
    StructField("quantity",IntegerType(),True)
])
df_data_empty=spark.createDataFrame(data_empty,schema)
def validate_data(df):
   return df.withColumn("status",when(col("quantity")>0,"Valid")
                                       .otherwise("Invalid"))
result_valid=validate_data(df_data_valid)
result_invalid=validate_data(df_data_invalid)
result_empty=validate_data(df_data_empty)
result_valid.show()
result_invalid.show()
result_empty.show()



+-----------+--------+------+
|customer_id|quantity|status|
+-----------+--------+------+
|        101|      10| Valid|
|        102|      25| Valid|
|        103|       5| Valid|
+-----------+--------+------+

+-----------+--------+-------+
|customer_id|quantity| status|
+-----------+--------+-------+
|        104|       0|Invalid|
|        105|      -5|Invalid|
|        106|    NULL|Invalid|
+-----------+--------+-------+

+-----------+--------+------+
|customer_id|quantity|status|
+-----------+--------+------+
+-----------+--------+------+



In [42]:
# Preserve source-file metadata so a quarantined line can be traced to its origin.
df_accepted=df_customers.filter((col("email").isNotNull()) & (trim("email") != '') & (col("city").isNotNull()))
df_accepted.show(truncate=False)
df_accepted.write.parquet("data/accepted",mode="overwrite")
df_accepted=spark.read.parquet("data/accepted")
df_quarantine = df_customers.filter((col("email").isNull()) | (trim("email") == '')  | (col("city").isNull()))
df_quarantine.show(truncate = False)
df_quarantine.write.parquet("data/quarantine",mode="overwrite")
df_quarantine=spark.read.parquet("data/quarantine")

+-----------+----------+-----------------+---------+----------+---------+
|customer_id|first_name|email            |city     |order_date|is_active|
+-----------+----------+-----------------+---------+----------+---------+
|101        |Ramya     |ramya@gmail.com  |Hyderbad |2024-01-15|true     |
|103        |Sita      |sita@gmail.com   |Chennai  |2024-03-10|false    |
|104        |Ravi      |ravi@gmail.com   |Mumbai   |2024-04-05|true     |
|101        |Priya     |priya@gmail.com  |Delhi    |2024-05-12|true     |
|105        |Kiran     |kiran39@gmail.com|Pune     |2024-06-18|false    |
|104        |Meena     |meena@gmail.com  |Kolkata  |2024-07-22|true     |
|109        |Divya     |divya@gmail.com  |Bangalore|2024-09-14|true     |
+-----------+----------+-----------------+---------+----------+---------+

+-----------+----------+-------------------+---------+----------+---------+
|customer_id|first_name|email              |city     |order_date|is_active|
+-----------+----------+---------

In [43]:
df_customers.show()

+-----------+----------+-------------------+---------+----------+---------+
|customer_id|first_name|              email|     city|order_date|is_active|
+-----------+----------+-------------------+---------+----------+---------+
|        101|     Ramya|    ramya@gmail.com| Hyderbad|2024-01-15|     true|
|        102|      Anil|               NULL|Bangalore|2024-02-20|     true|
|        103|      Sita|     sita@gmail.com|  Chennai|2024-03-10|    false|
|        104|      Ravi|     ravi@gmail.com|   Mumbai|2024-04-05|     true|
|        101|     Priya|    priya@gmail.com|    Delhi|2024-05-12|     true|
|        105|     Kiran|  kiran39@gmail.com|     Pune|2024-06-18|    false|
|        104|     Meena|    meena@gmail.com|  Kolkata|2024-07-22|     true|
|        108|     Arjun|                   |Hyderabad|2024-08-09|     true|
|        109|     Divya|    divya@gmail.com|Bangalore|2024-09-14|     true|
|        103|     Varun|varun400@outook.com|     NULL|2024-10-25|    false|
+-----------

In [70]:
# Preserve source-file metadata so a quarantined line can be traced to its origin.
df_customers.write.csv("data.csv/customers.csv",header=True,mode="overwrite")
df_customers_file=spark.read.csv("data.csv/customers.csv",header=True)
df_customers_file=df_customers_file.withColumn("input_file_name",input_file_name())\
                         .withColumn("current_timestamp",current_timestamp())
df_customers_file.show(truncate=False)
df_accepted=df_customers_file.filter((col("email").isNotNull()) & (trim("email")!="") & (col("city").isNotNull()))
df_accepted.write.parquet("data.parquet/df_accepted",mode="overwrite")
df_quarentine=df_customers_file.filter((col("email").isNull()) | (trim("email")=="") | (col("city").isNull()))
df_quarentine.write.parquet("data.parquet/df_quarentine",mode="overwrite")
df_quarentine_saved=spark.read.parquet("data.parquet/df_quarentine")
df_quarentine_saved.show(truncate=False)


+-----------+----------+-------------------+---------+----------+---------+-----------------------------------------------------------------------------------------------+--------------------------+
|customer_id|first_name|email              |city     |order_date|is_active|input_file_name                                                                                |current_timestamp         |
+-----------+----------+-------------------+---------+----------+---------+-----------------------------------------------------------------------------------------------+--------------------------+
|105        |Kiran     |KIRAN39@GMAIL.COM  |Pune     |2024-06-18|false    |file:///content/data.csv/customers.csv/part-00001-0ea314a7-f9fa-486a-9e4d-879cc8166feb-c000.csv|2026-10-09 07:09:10.078907|
|104        |Meena     |meena@gmail.com    |KOLKATA  |2024-07-22|true     |file:///content/data.csv/customers.csv/part-00001-0ea314a7-f9fa-486a-9e4d-879cc8166feb-c000.csv|2026-10-09 07:09:10.078907|
|108 

In [73]:
total_count = df_customers_file.count()
accepted_count = df_accepted.count()
quarantined_count = df_quarentine.count()

rejection_rate = (
    quarantined_count / total_count * 100
    if total_count > 0 else 0
)
print(rejection_rate)
max_rejection_threshold = 5

if rejection_rate > max_rejection_threshold:
    print("Pipeline FAILED: Rejection threshold exceeded")
else:
    print("Pipeline PASSED: Rejection threshold satisfied")

30.0
Pipeline FAILED: Rejection threshold exceeded


#Define a maximum rejection threshold and a pipeline failure policy.

1. Maximum Rejection Threshold: The maximum allowable percentage of rejected records in a pipeline before it is considered a failure. For example, set the threshold at 5%.

2. Pipeline Failure Policy: The rules that determine how the pipeline responds when the rejection threshold is exceeded or a critical error occurs. For example, stop downstream processing, log the error, preserve rejected records, and retry if the failure is temporary.

#Design a correction-and-reprocessing workflow that preserves the original audit.

Correction-and-Reprocessing Workflow That Preserves the Original Audit

When records are rejected and moved to quarantine, correct the invalid data and reprocess it without modifying or deleting the original records or audit history.

Preserve the original data: Store the original input file, timestamp, file name, pipeline run ID, and rejection reason.

Identify and correct errors: Fix invalid records in a separate copy of the quarantined data.

Reprocess corrected records: Run the corrected records through the validation pipeline again.

Record the results: Track each reprocessing attempt, timestamp, and final status (accepted or quarantined).

Maintain the audit trail: Keep the original records and rejection details unchanged, and record all corrections and reprocessing results separately.

Handle failures: If records fail validation again, keep them quarantined for further correction or manual review.



In [ ]:
#Implement a reusable validator returning accepted rows, quarantine rows, and metrics.

total_count = df_customers_file.count()
accepted_count = df_accepted.count()
quarantined_count = df_quarentine.count()

rejection_rate = (
    quarantined_count / total_count * 100
    if total_count > 0 else 0
)
print(rejection_rate)
max_rejection_threshold = 5

if rejection_rate > max_rejection_threshold:
    print("Pipeline FAILED: Rejection threshold exceeded")
else:
    print("Pipeline PASSED: Rejection threshold satisfied")